### Content-Based Recommentdation

##### Ví dụ:
- Người dùng thích: GTA V
- -> Hệ thống sẽ recommend:
    + Red Dead Redemption
    + Watch Dogs
    + Mafia
- Dựa trên:
    + genres
    + categories
    + description
    + tags
    + metacritic
    + platform

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#### 1. Load Dataset

In [3]:
df = pd.read_csv(
    "../data/raw/applications.csv",
    low_memory=False
)

print(df.shape)

df.head()

(239664, 30)


,appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,...,mat_pc_os_min,mat_pc_processor_min,mat_pc_memory_min,mat_pc_graphics_min,mat_pc_os_rec,mat_pc_processor_rec,mat_pc_memory_rec,mat_pc_graphics_rec,created_at,updated_at
0,10,Counter-Strike,game,False,2000-11-01,0,Play the world's number 1 online action game. ...,"English<strong>*</strong>, French<strong>*</st...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
1,20,Team Fortress Classic,game,False,1999-04-01,0,One of the most popular online action games of...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
2,30,Day of Defeat,game,False,2003-05-01,0,Enlist in an intense brand of Axis vs. Allied ...,"English, French, German, Italian, Spanish - Spain",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
3,40,Deathmatch Classic,game,False,2001-06-01,0,Enjoy fast-paced multiplayer gaming with Death...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
4,50,Half-Life: Opposing Force,game,False,1999-11-01,0,Return to the Black Mesa Research Facility as ...,"English, French, German, Korean",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00


#### 2. Chọn Feature Recommendation

- Dùng: name + short_description

In [ ]:
df_rec = df[[
    'appid',
    'name',
    'type',
    'short_description'
]].copy()

#### 3. Missing Values

In [5]:
df_rec = df_rec.dropna()

df_rec.reset_index(drop=True, inplace=True)

#### 4. Clean text

In [8]:
df_rec['short_description'] = (
    df_rec['short_description']
    .str.lower()
    .str.replace(r'<.*?>', '', regex=True)
)

#### 5. TF-IDF

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(
    df_rec['short_description']
)

print(tfidf_matrix.shape)

(224170, 5000)


#### 6. Hàm recommendation realtime

In [12]:
from sklearn.metrics.pairwise import linear_kernel

def recommend_games(game_name, top_n=5):

    idx_list = df_rec[
        df_rec['name'].str.contains(game_name, case=False, na=False)
    ].index

    if len(idx_list) == 0:
        print("Không tìm thấy game.")
        return

    idx = idx_list[0]

    cosine_scores = linear_kernel(
        tfidf_matrix[idx:idx+1],
        tfidf_matrix
    ).flatten()

    similar_indices = cosine_scores.argsort()[-top_n-1:-1][::-1]

    recommendations = df_rec.iloc[similar_indices][
        ['name', 'type']
    ]

    return recommendations

#### 7. Test hệ khuyến nghị

In [17]:
recommend_games("Counter-Strike")

,name,type
90931,Mind-Blowing Girls 2 Arts,dlc
12506,GabeN: The Final Decision,game
24087,"Planes, Bullets and Vodka: Soundtrack",dlc
51744,Cell Defender,game
75372,Catch It,game


#### 8. Final Conclusion

Nhóm đã xây dựng hệ khuyến nghị game dựa trên nội dung (Content-Based Recommendation System) sử dụng mô tả ngắn của các trò chơi trên Steam.

- Quy trình thực hiện gồm:
    + Tiền xử lý văn bản (Text Preprocessing)
    + TF-IDF Vectorization
    + Tính độ tương đồng bằng Cosine Similarity
    + Đề xuất các game có nội dung tương tự

- Do dataset có kích thước rất lớn (hơn 220.000 game), việc tính toàn bộ ma trận similarity gây vượt quá giới hạn bộ nhớ RAM. Vì vậy, nhóm sử dụng phương pháp tính similarity động bằng linear_kernel() chỉ cho game được truy vấn thay vì tính toàn bộ ma trận.

- Kết quả đạt được:
    + Đề xuất các game có nội dung tương đồng khá hợp lý
    + Hệ thống hoạt động hiệu quả trên dữ liệu lớn
    + Có khả năng mở rộng cho các hệ recommendation thực tế

- Hệ khuyến nghị là một trong những ứng dụng quan trọng của Machine Learning và Data Science trong:
    + Thương mại điện tử
    + Nền tảng phân phối game số
    + Hệ thống gợi ý sản phẩm và nội dung số hiện đại